# Exercice 1 — Exploration des relevés d'incidents

## Objectif et point de départ

Ce notebook analyse les données saisies par les opérateurs d'une usine fonctionnant 24 h/24 selon le rythme des trois-huit. Le fichier source Bronze `datas/releves_incidents.csv.csv` est conservé sans modification.

La cellule suivante importe les outils nécessaires, vérifie que le fichier existe, puis le charge dans un **DataFrame**, c'est-à-dire un tableau pandas composé de lignes et de colonnes nommées.

In [1]:
from pathlib import Path

import pandas as pd
from tabulate import tabulate

# Le chemin est relatif au dossier du notebook : indusense/.
chemin_incidents = Path("datas") / "releves_incidents.csv.csv"

# Une erreur explicite évite de poursuivre avec un mauvais emplacement.
if not chemin_incidents.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {chemin_incidents.resolve()}")

# read_csv lit le fichier source sans le modifier et crée le DataFrame en mémoire.
incidents_bruts = pd.read_csv(chemin_incidents)

print(f"Fichier chargé : {chemin_incidents.resolve()}")
print("Chargement réussi.")

Fichier chargé : D:\source\A4U\FormationIA\indusense\datas\releves_incidents.csv.csv
Chargement réussi.


## 1. Extraire la structure et les dix premières lignes

La cellule suivante utilise `shape` pour obtenir les dimensions du tableau, `columns` pour récupérer les en-têtes et `head(10)` pour sélectionner les dix premières lignes. `tabulate` affiche l'aperçu sous forme de tableau en texte brut, sans HTML.

In [2]:
# shape renvoie un tuple : nombre de lignes, puis nombre de colonnes.
nombre_lignes, nombre_colonnes = incidents_bruts.shape
en_tetes = incidents_bruts.columns.tolist()
dix_premieres_lignes = incidents_bruts.head(10)

print(f"Nombre de lignes   : {nombre_lignes}")
print(f"Nombre de colonnes : {nombre_colonnes}")
print("En-têtes de colonnes :")
for position, nom_colonne in enumerate(en_tetes, start=1):
    print(f"  {position:>2}. {nom_colonne}")

print("\nDix premières lignes :")
print(tabulate(dix_premieres_lignes, headers="keys", tablefmt="fancy_grid", showindex=False))

Nombre de lignes   : 1245
Nombre de colonnes : 18
En-têtes de colonnes :
   1. incident_id
   2. date
   3. time
   4. operator_name
   5. machine_id
   6. severity
   7. operator_badge
   8. comment
   9. shift
  10. type_surchauffe
  11. type_baisse_pression
  12. type_vibration
  13. type_bruit_mecanique
  14. type_surconsommation
  15. type_blocage_mecanique
  16. type_alarme_capteur
  17. type_arret_urgence
  18. type_defaut_qualite

Dix premières lignes :
╒═══════════════╤════════════╤════════╤═════════════════╤══════════════╤════════════╤══════════════════╤═══════════════════════════════╤════════════╤═══════════════════╤════════════════════════╤══════════════════╤════════════════════════╤════════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════╤═══════════════════════╕
│ incident_id   │ date       │ time   │ operator_name   │ machine_id   │   severity │ operator_badge   │ comment                       │ shift      │   type_surchauffe │   

## 2a. Observer les types détectés automatiquement

Un **type de données** décrit la nature des valeurs et les opérations possibles. La cellule suivante affiche le type attribué par pandas lors de la lecture initiale. Cette première observation permet notamment de voir que la date n'est pas encore reconnue comme une date exploitable.

In [3]:
# dtypes contient le type pandas détecté pour chaque colonne.
types_inferes = incidents_bruts.dtypes.rename("type pandas inféré").reset_index()
types_inferes.columns = ["colonne", "type pandas inféré"]

print(tabulate(types_inferes, headers="keys", tablefmt="fancy_grid", showindex=False))

╒════════════════════════╤══════════════════════╕
│ colonne                │ type pandas inféré   │
╞════════════════════════╪══════════════════════╡
│ incident_id            │ str                  │
├────────────────────────┼──────────────────────┤
│ date                   │ str                  │
├────────────────────────┼──────────────────────┤
│ time                   │ str                  │
├────────────────────────┼──────────────────────┤
│ operator_name          │ str                  │
├────────────────────────┼──────────────────────┤
│ machine_id             │ str                  │
├────────────────────────┼──────────────────────┤
│ severity               │ int64                │
├────────────────────────┼──────────────────────┤
│ operator_badge         │ str                  │
├────────────────────────┼──────────────────────┤
│ comment                │ str                  │
├────────────────────────┼──────────────────────┤
│ shift                  │ str                  │


## 2b. Donner aux colonnes des types adaptés — deuxième itération

**Première itération :** `convert_dtypes()` a correctement reconnu les textes et les entiers, mais a conservé les colonnes `type_*` en `Int64`. Ce type décrit leur stockage sous forme de `0` et de `1`, pas leur signification métier.

**Relecture et correction :** ces colonnes répondent à une question binaire — le type d'incident est présent ou absent. Leur type adapté est donc `boolean`, avec `False` pour `0` et `True` pour `1`.

La cellule suivante crée une copie typée, vérifie d'abord que les colonnes `type_*` ne contiennent que `0` ou `1`, les convertit en `boolean`, puis convertit `date` en `datetime64`. L'option `errors="raise"` arrête immédiatement l'analyse si une date est invalide. Le CSV Bronze reste inchangé : seule la copie en mémoire est transformée.

In [4]:
# La copie évite de confondre les données chargées avec leur version typée.
incidents = incidents_bruts.convert_dtypes()
incidents["date"] = pd.to_datetime(incidents["date"], errors="raise")

# Les colonnes type_* expriment une présence ou une absence : leur type métier est booléen.
colonnes_types_incident = [colonne for colonne in incidents.columns if colonne.startswith("type_")]
for colonne in colonnes_types_incident:
    valeurs_observees = set(incidents[colonne].dropna().unique())
    if not valeurs_observees.issubset({0, 1}):
        raise ValueError(f"La colonne {colonne} contient une valeur différente de 0 ou 1.")
    incidents[colonne] = incidents[colonne].astype("boolean")

# Une interprétation métier complète le nom technique du type pandas.
interpretations = {
    "incident_id": "identifiant textuel d'incident",
    "date": "date calendaire",
    "time": "heure représentée sous forme de texte",
    "operator_name": "nom d'opérateur",
    "machine_id": "identifiant textuel de machine",
    "severity": "niveau de sévérité entier",
    "operator_badge": "identifiant textuel d'opérateur",
    "comment": "commentaire textuel",
    "shift": "équipe des trois-huit",
}

# Après conversion, False signifie absent et True signifie présent.
for colonne in colonnes_types_incident:
    interpretations[colonne] = "indicateur booléen d'un type d'incident"

rapport_types = pd.DataFrame({
    "colonne": incidents.columns,
    "type pandas": incidents.dtypes.astype(str).to_numpy(),
    "interprétation": [interpretations[colonne] for colonne in incidents.columns],
})

print(tabulate(rapport_types, headers="keys", tablefmt="fancy_grid", showindex=False))

╒════════════════════════╤════════════════╤═════════════════════════════════════════╕
│ colonne                │ type pandas    │ interprétation                          │
╞════════════════════════╪════════════════╪═════════════════════════════════════════╡
│ incident_id            │ string         │ identifiant textuel d'incident          │
├────────────────────────┼────────────────┼─────────────────────────────────────────┤
│ date                   │ datetime64[us] │ date calendaire                         │
├────────────────────────┼────────────────┼─────────────────────────────────────────┤
│ time                   │ string         │ heure représentée sous forme de texte   │
├────────────────────────┼────────────────┼─────────────────────────────────────────┤
│ operator_name          │ string         │ nom d'opérateur                         │
├────────────────────────┼────────────────┼─────────────────────────────────────────┤
│ machine_id             │ string         │ identifian

## 3. Calculer les valeurs minimales et maximales

La cellule suivante applique `min()` et `max()` à `severity` et à `date`. Pour la sévérité, ces opérations trouvent les niveaux extrêmes. Pour la date typée, elles trouvent les incidents les plus anciens et les plus récents.

In [5]:
# Les calculs sont conservés dans des variables pour être réutilisés dans le rapport final.
severite_min = incidents["severity"].min()
severite_max = incidents["severity"].max()
date_min = incidents["date"].min()
date_max = incidents["date"].max()

rapport_extremes = [
    ["severity", severite_min, severite_max],
    ["date", date_min.date().isoformat(), date_max.date().isoformat()],
]

print(tabulate(rapport_extremes, headers=["colonne", "minimum", "maximum"], tablefmt="fancy_grid"))

╒═══════════╤════════════╤════════════╕
│ colonne   │ minimum    │ maximum    │
╞═══════════╪════════════╪════════════╡
│ severity  │ 2          │ 5          │
├───────────┼────────────┼────────────┤
│ date      │ 2025-06-01 │ 2026-06-08 │
╘═══════════╧════════════╧════════════╛


## 4. Compter les machines, les opérateurs et les commentaires distincts

Une valeur **distincte** n'est comptée qu'une seule fois, même si elle apparaît sur plusieurs incidents. La cellule suivante utilise `nunique()`. La consigne portant sur les opérateurs, `operator_name` sert au calcul principal. Le nombre de badges est aussi calculé comme contrôle de qualité : l'écart entre les deux résultats signale que les badges ne permettent pas d'identifier seuls les personnes dans ce fichier.

In [6]:
# nunique exclut par défaut les valeurs manquantes ; dropna=True rend ce choix explicite.
nombre_machines = incidents["machine_id"].nunique(dropna=True)
nombre_operateurs = incidents["operator_name"].nunique(dropna=True)
nombre_badges = incidents["operator_badge"].nunique(dropna=True)
nombre_commentaires = incidents["comment"].nunique(dropna=True)

rapport_distincts = [
    ["Machines distinctes (machine_id)", nombre_machines],
    ["Opérateurs distincts (operator_name)", nombre_operateurs],
    ["Badges distincts (contrôle qualité)", nombre_badges],
    ["Commentaires distincts", nombre_commentaires],
]

print(tabulate(rapport_distincts, headers=["indicateur", "valeur"], tablefmt="fancy_grid"))
if nombre_badges != nombre_operateurs:
    print("\nAlerte qualité : le nombre de badges diffère du nombre de noms d'opérateurs.")

╒══════════════════════════════════════╤══════════╕
│ indicateur                           │   valeur │
╞══════════════════════════════════════╪══════════╡
│ Machines distinctes (machine_id)     │       15 │
├──────────────────────────────────────┼──────────┤
│ Opérateurs distincts (operator_name) │       15 │
├──────────────────────────────────────┼──────────┤
│ Badges distincts (contrôle qualité)  │       10 │
├──────────────────────────────────────┼──────────┤
│ Commentaires distincts               │       46 │
╘══════════════════════════════════════╧══════════╛

Alerte qualité : le nombre de badges diffère du nombre de noms d'opérateurs.


## 5. Afficher le rapport complet en texte brut

La cellule suivante rassemble tous les résultats demandés dans une sortie unique et visible. `print()` reçoit des chaînes produites par `tabulate` avec le style `fancy_grid` : le résultat reste du texte et ne dépend d'aucun rendu HTML.

In [7]:
# Les séparateurs rendent les différentes parties du rapport faciles à repérer.
separateur = "=" * 110

print(separateur)
print("RAPPORT COMPLET — RELEVÉS D'INCIDENTS")
print(separateur)
print(f"Source              : {chemin_incidents.resolve()}")
print(f"Nombre de lignes    : {nombre_lignes}")
print(f"Nombre de colonnes  : {nombre_colonnes}")

print("\n1. EN-TÊTES")
table_en_tetes = [[position, nom] for position, nom in enumerate(en_tetes, start=1)]
print(tabulate(table_en_tetes, headers=["position", "colonne"], tablefmt="fancy_grid"))

print("\n2. TYPES DES COLONNES")
print(tabulate(rapport_types, headers="keys", tablefmt="fancy_grid", showindex=False))

print("\n3. VALEURS MINIMALES ET MAXIMALES")
print(tabulate(rapport_extremes, headers=["colonne", "minimum", "maximum"], tablefmt="fancy_grid"))

print("\n4. NOMBRES DE VALEURS DISTINCTES")
print(tabulate(rapport_distincts, headers=["indicateur", "valeur"], tablefmt="fancy_grid"))

print("\n5. DIX PREMIÈRES LIGNES")
print(tabulate(dix_premieres_lignes, headers="keys", tablefmt="fancy_grid", showindex=False))
print(separateur)

RAPPORT COMPLET — RELEVÉS D'INCIDENTS
Source              : D:\source\A4U\FormationIA\indusense\datas\releves_incidents.csv.csv
Nombre de lignes    : 1245
Nombre de colonnes  : 18

1. EN-TÊTES
╒════════════╤════════════════════════╕
│   position │ colonne                │
╞════════════╪════════════════════════╡
│          1 │ incident_id            │
├────────────┼────────────────────────┤
│          2 │ date                   │
├────────────┼────────────────────────┤
│          3 │ time                   │
├────────────┼────────────────────────┤
│          4 │ operator_name          │
├────────────┼────────────────────────┤
│          5 │ machine_id             │
├────────────┼────────────────────────┤
│          6 │ severity               │
├────────────┼────────────────────────┤
│          7 │ operator_badge         │
├────────────┼────────────────────────┤
│          8 │ comment                │
├────────────┼────────────────────────┤
│          9 │ shift                  │
├──────

## Bonus — Moyennes et médianes de la télémétrie

Une **moyenne** est la somme des valeurs divisée par leur nombre. Une **médiane** partage les observations ordonnées en deux groupes de même taille et résiste mieux aux valeurs extrêmes. La cellule suivante charge `telemetry.csv.csv`, convertit son horodatage et calcule ces deux statistiques pour chaque mesure numérique.

In [8]:
chemin_telemetrie = Path("datas") / "telemetry.csv.csv"

# Le bonus reste indépendant : une absence du fichier produit une erreur claire.
if not chemin_telemetrie.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {chemin_telemetrie.resolve()}")

# parse_dates convertit timestamp dès la lecture du CSV.
telemetrie = pd.read_csv(chemin_telemetrie, parse_dates=["timestamp"]).convert_dtypes()
colonnes_mesures = [
    "temperature_c",
    "pressure_bar",
    "voltage_mean_v",
    "rotation_mean_rpm",
    "pieces_produced",
]

# agg applique les deux fonctions à toutes les colonnes de mesure sélectionnées.
statistiques_telemetrie = (
    telemetrie[colonnes_mesures]
    .agg(["mean", "median"])
    .transpose()
    .reset_index(names="mesure")
)

print(f"Lignes de télémétrie : {len(telemetrie)}")
print(f"Machines distinctes : {telemetrie['machine_id'].nunique()}")
print(tabulate(statistiques_telemetrie, headers="keys", tablefmt="fancy_grid", showindex=False, floatfmt=".3f"))

Lignes de télémétrie : 135626
Machines distinctes : 15
╒═══════════════════╤══════════╤══════════╕
│ mesure            │     mean │   median │
╞═══════════════════╪══════════╪══════════╡
│ temperature_c     │   48.183 │   48.055 │
├───────────────────┼──────────┼──────────┤
│ pressure_bar      │  199.770 │  199.866 │
├───────────────────┼──────────┼──────────┤
│ voltage_mean_v    │  227.631 │  227.420 │
├───────────────────┼──────────┼──────────┤
│ rotation_mean_rpm │ 1589.205 │ 1590.370 │
├───────────────────┼──────────┼──────────┤
│ pieces_produced   │   49.533 │   49.000 │
╘═══════════════════╧══════════╧══════════╛


## Validation finale

La dernière cellule effectue des **assertions**, c'est-à-dire des contrôles qui provoquent une erreur si une condition attendue est fausse. Elle vérifie la présence des colonnes demandées, le typage de la date, la cohérence des bornes et des comptages, ainsi que les calculs du bonus. Le message final ne s'affiche que si tous les contrôles réussissent.

In [9]:
# Ces contrôles sécurisent les hypothèses utilisées dans le rapport.
colonnes_obligatoires = {"date", "severity", "machine_id", "operator_badge", "comment"}
assert colonnes_obligatoires.issubset(incidents.columns), "Une colonne obligatoire manque."
assert pd.api.types.is_datetime64_any_dtype(incidents["date"]), "La date n'est pas correctement typée."
assert all(pd.api.types.is_bool_dtype(incidents[colonne]) for colonne in colonnes_types_incident), (
    "Toutes les colonnes type_* doivent être booléennes."
)
assert len(dix_premieres_lignes) == min(10, nombre_lignes), "L'aperçu ne contient pas le nombre attendu de lignes."
assert severite_min <= severite_max, "Les bornes de sévérité sont incohérentes."
assert date_min <= date_max, "Les bornes de date sont incohérentes."
assert nombre_machines <= nombre_lignes, "Le nombre de machines est incohérent."
assert nombre_operateurs <= nombre_lignes, "Le nombre d'opérateurs est incohérent."
assert nombre_commentaires <= nombre_lignes, "Le nombre de commentaires est incohérent."
assert statistiques_telemetrie[["mean", "median"]].notna().all().all(), "Une statistique de télémétrie manque."

print("Validation réussie : toutes les cellules ont produit des résultats cohérents.")

Validation réussie : toutes les cellules ont produit des résultats cohérents.
